# Stage 29A: multi-cut strong-base residual field

Stage 28Aと同じ500 training wellsから全安全short-prefix cutsを使います。モデルと固定weight 0.20は変更しません。CPU・ハイメモリ推奨、予約120 wellsは未使用です。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
print('CPU high-memory runtime recommended; GPU is unused')


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet']
for path in required: assert path.is_file(),path


In [ ]:
BASE_ID='stage28a_split_manifest_v001'; base_manifest=artifact_dir/BASE_ID
base_files=[base_manifest/'summary.json',base_manifest/'training_cut_ids.parquet',base_manifest/'confirmation_cut_ids.parquet']
if not all(path.is_file() for path in base_files):
    if base_manifest.exists():
        resolved=base_manifest.resolve(); expected=(artifact_dir/BASE_ID).resolve(); assert resolved==expected and resolved.parent==artifact_dir.resolve(); shutil.rmtree(resolved)
    command=['uv','run','rogii-scaled-emission-manifest','--stage17a-run',str(stage17a_run),'--design-validation-run',str(stage21b_run),'--artifact-dir',str(artifact_dir),'--run-id',BASE_ID]
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True); print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError(command)
MANIFEST_ID='stage29a_multicut_manifest_v001'; manifest_run=artifact_dir/MANIFEST_ID
manifest_files=[manifest_run/'summary.json',manifest_run/'training_cut_ids.parquet',manifest_run/'confirmation_cut_ids.parquet']
if not all(path.is_file() for path in manifest_files):
    if manifest_run.exists():
        resolved=manifest_run.resolve(); expected=(artifact_dir/MANIFEST_ID).resolve(); assert resolved==expected and resolved.parent==artifact_dir.resolve(); shutil.rmtree(resolved)
    command=['uv','run','rogii-expanded-residual-manifest','--stage17a-run',str(stage17a_run),'--base-manifest-run',str(base_manifest),'--artifact-dir',str(artifact_dir),'--run-id',MANIFEST_ID]
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True); print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError(command)
manifest=json.loads((manifest_run/'summary.json').read_text())
assert manifest['training_wells']==500 and manifest['confirmation_wells']==120 and manifest['reserved_confirmation_used'] is False,manifest
manifest


In [ ]:
RUN_ID='stage29a_multicut_residual_field_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve(); assert resolved==expected and resolved.parent==artifact_dir.resolve(); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=['uv','run','rogii-expanded-residual-field','--config','configs/experiment/stage29a_multicut_residual_field.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--split-manifest-run',str(manifest_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID]
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f'Stage 29A failed: {command}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage29a_complete','promoted_to_stage29b_reserved_confirmation','training_cuts','training_wells','validation_cuts','validation_wells','training_rows','feature_count','ensemble_models','primary_weight','base_rmse','candidate_rmse','rmse_delta','bootstrap_95pct','cut_rmse_p90_delta','gates','reserved_confirmation_used','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['weight_report']).sort_values('weight')
